## **MLlib: 영화 추천 알고리즘 구현**
+ 데이터셋: MovieLens(2500만개 영화 평점 데이터), 본 예제에서는 7만개의 데이터만 사용 ```ratings_short.csv```
+ 추천 알고리즘: Alternating Least Squares (ALS)

---

### **1. 영화 평점 데이터 불러오기 & 데이터프레임 생성하기**

In [4]:
# [+] SparkSession 설정
from pyspark.sql import SparkSession
spark = (
    SparkSession
    .builder
    .master('local')
    .appName('recommendation')
    .getOrCreate()
)

#### movielens 데이터 불러오기
+ 파일명: ```ratings_short.csv```
+ 스키마 설정: ```inferSchema=True```
+ 헤더 사용: ```header=True```

In [5]:
# movielens 데이터 불러오기
path = '../data/'
file = 'ratings_short.csv'

ratings_df = spark.read.csv(
    path + file, 
    inferSchema=True, 
    header=True
)

In [6]:
# 데이터프레임 출력
ratings_df.show()

+------+-------+------+----------+
|userId|movieId|rating| timestamp|
+------+-------+------+----------+
|     1|    296|   5.0|1147880044|
|     1|    306|   3.5|1147868817|
|     1|    307|   5.0|1147868828|
|     1|    665|   5.0|1147878820|
|     1|    899|   3.5|1147868510|
|     1|   1088|   4.0|1147868495|
|     1|   1175|   3.5|1147868826|
|     1|   1217|   3.5|1147878326|
|     1|   1237|   5.0|1147868839|
|     1|   1250|   4.0|1147868414|
|     1|   1260|   3.5|1147877857|
|     1|   1653|   4.0|1147868097|
|     1|   2011|   2.5|1147868079|
|     1|   2012|   2.5|1147868068|
|     1|   2068|   2.5|1147869044|
|     1|   2161|   3.5|1147868609|
|     1|   2351|   4.5|1147877957|
|     1|   2573|   4.0|1147878923|
|     1|   2632|   5.0|1147878248|
|     1|   2692|   5.0|1147869100|
+------+-------+------+----------+
only showing top 20 rows



In [7]:
# 데이터프레임 스키마 출력
ratings_df.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [10]:
# [+] 타임스탬프를 제외한 컬럼 선택
ratings_df = ratings_df.select(['userId', 'movieId', 'rating'])

In [12]:
# [+] describe(): 기술 통계량 출력
ratings_df.select('rating').describe().show()

[Stage 5:>                                                          (0 + 1) / 1]

+-------+------------------+
|summary|            rating|
+-------+------------------+
|  count|             71921|
|   mean|3.5821387355570695|
| stddev| 1.042406032579843|
|    min|               0.5|
|    max|               5.0|
+-------+------------------+



### **2. 훈련 데이터 준비 및 영화 추천 모델 학습**

In [13]:
# [+] 훈련 데이터셋과 테스트 데이터셋 분리 (8:2)
train_df, test_df = ratings_df.randomSplit([0.8, 0.2])

In [14]:
# 추천 알고리즘(Alternating Least Squares) 불러오기
from  pyspark.ml.recommendation import ALS

#### 
**ALS 알고리즘 초매개변수**
+ ```maxIter```: 최대 학습 반복 횟수
+ ```regParam```: 정규화 매개변수(범위: 0~1)
+ ```coldStartStrategy```: 데이터가 부족한 신규 유저 및 아이템에 대한 예측 문제(Cold Start)를 처리하는 방식이며 ```drop```값은 해당 데이터를 모델 학습 과정에서 배제

In [15]:
# 추천 알고리즘 초매개변수 설정
als = ALS(
    maxIter=5,
    regParam=0.1,
    userCol='userId',
    itemCol='movieId',
    ratingCol='rating',
    coldStartStrategy='drop'
)

In [18]:
# [+] 모델 학습
model = als.fit(train_df) # 학습

In [19]:
# # 메모리 부족으로 인한 오류 발생시, 아래의 코드를 실행
# from pyspark.sql import SparkSession

# MAX_MEMORY = '5g'
# spark = SparkSession.builder.appName('movie-recommendation')\
#     .config('spark.executor.memory', MAX_MEMORY)\
#     .config('spark.driver.memory', MAX_MEMORY)\
#     .getOrCreate()

In [20]:
# [+] 모델 예측
predictions = model.transform(test_df)

In [21]:
# 예측값 출력
predictions.show()

+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|   148|     47|   4.0| 4.1443768|
|   148|    293|   4.0| 4.1226044|
|   148|    904|   4.5|  3.992259|
|   148|    924|   5.0|  4.097372|
|   148|    926|   4.0| 3.9135854|
|   148|   1086|   4.0| 4.1443486|
|   148|   1198|   4.0|  4.152819|
|   148|   1204|   4.0| 4.2067146|
|   148|   1617|   4.0| 4.0577703|
|   148|   2019|   5.0|  4.163574|
|   148|   2028|   4.0| 3.7692404|
|   148|   3307|   4.0| 4.4071093|
|   148|  48394|   4.0| 4.0860896|
|   148|  48516|   4.5|  4.126611|
|   148|  77455|   3.5| 2.9076872|
|   148|  81834|   3.5| 3.5597057|
|   148|  88810|   4.0| 3.7207005|
|   148| 109487|   4.0| 4.0574207|
|   463|    788|   4.0| 3.6522565|
|   471|    318|   4.0|  4.386537|
+------+-------+------+----------+
only showing top 20 rows



In [22]:
# [+] 평점과 예측평점에 대한 통계 출력
predictions.select(['rating', 'prediction']).describe().show()

[Stage 177:>                                                        (0 + 1) / 1]

+-------+------------------+------------------+
|summary|            rating|        prediction|
+-------+------------------+------------------+
|  count|             13592|             13592|
|   mean|3.6126765744555622|3.4176034708575878|
| stddev|1.0354323801360705|0.7548867894274328|
|    min|               0.5|        0.23832184|
|    max|               5.0|          5.695086|
+-------+------------------+------------------+



In [23]:
# 모델 성능 평가: RMSE(Root Mean Squared Error)
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='rating',
    predictionCol='prediction'
)

In [29]:
# [+] RMSE 측정
mae = evaluator.evaluate(predictions)

In [30]:
mae

0.918857184734672

### **3. 학습된 모델을 이용한 영화 추천**
+ ```recommendForAllUsers()```: 유저별 아이템 추천
+ ```recommendForAllItems()```: 아이템별 유저 추천
+ ```recommendForUserSubset()```: 특정 유저 그룹에 대한 아이템 추천

In [31]:
# [+] 유저별 아이템을 3개씩 추천
model.recommendForAllUsers(3).show()

[Stage 299:====================================================> (98 + 1) / 100]

+------+--------------------+
|userId|     recommendations|
+------+--------------------+
|     1|[{8327, 5.026992}...|
|     2|[{117364, 5.62876...|
|     3|[{142871, 5.14182...|
|     4|[{8607, 5.045233}...|
|     5|[{117364, 5.69720...|
|     6|[{1484, 5.357182}...|
|     7|[{1238, 4.747985}...|
|     8|[{177765, 5.08554...|
|     9|[{5152, 5.614417}...|
|    10|[{117364, 5.12409...|
|    11|[{117364, 4.19057...|
|    12|[{1243, 4.630552}...|
|    13|[{142871, 5.09221...|
|    14|[{3881, 5.5311356...|
|    15|[{101070, 6.12638...|
|    16|[{142871, 5.26209...|
|    17|[{1627, 4.729201}...|
|    18|[{66785, 4.939393...|
|    19|[{141432, 4.88114...|
|    20|[{142871, 5.66145...|
+------+--------------------+
only showing top 20 rows



In [33]:
# [+] 아이템별 유저를 3명씩 추천
model.recommendForAllItems(3).show()

[Stage 357:====================================================> (98 + 1) / 100]

+-------+--------------------+
|movieId|     recommendations|
+-------+--------------------+
|      1|[{327, 5.0131435}...|
|      2|[{474, 4.5938272}...|
|      3|[{210, 4.7632375}...|
|      4|[{474, 4.7726007}...|
|      5|[{173, 4.439574},...|
|      6|[{240, 5.2996984}...|
|      7|[{210, 4.8607106}...|
|      8|[{474, 4.3339534}...|
|      9|[{247, 3.7853246}...|
|     10|[{173, 4.5894647}...|
|     11|[{474, 5.0761027}...|
|     12|[{198, 4.64536}, ...|
|     13|[{153, 5.573419},...|
|     14|[{240, 4.7375093}...|
|     15|[{474, 4.388783},...|
|     16|[{87, 5.482142}, ...|
|     17|[{484, 5.390043},...|
|     18|[{240, 5.1358895}...|
|     19|[{240, 4.6829786}...|
|     20|[{456, 4.1819787}...|
+-------+--------------------+
only showing top 20 rows



**특정 유저 그룹에 대한 아이템 추천**
1. 해당 유저들의 ID 정보를 데이터프레임으로 생성
2. ```recommendForUserSubset()```를 이용한 아이템 추천

In [36]:
# 특정 유저 ID 집합: 2차원 리스트
user_lst = [[1]]

In [37]:
# 스키마 정의
from pyspark.sql.types import StructType, StructField, IntegerType
schema = StructType([
    StructField('userID', IntegerType(), True)
])

In [38]:
# 데이터프레임 생성
users_df = spark.createDataFrame(user_lst, schema)

In [39]:
users_df.show()

[Stage 373:>                                                        (0 + 1) / 1]

+------+
|userID|
+------+
|     1|
+------+



In [40]:
# [+] recommendForUserSubset(): 특정 유저 그룹에 대한 아이템 추천
user_recs = model.recommendForUserSubset(users_df, 5)

In [41]:
user_recs.show()

[Stage 417:==================================================>     (9 + 1) / 10]

+------+--------------------+
|userId|     recommendations|
+------+--------------------+
|     1|[{8327, 5.026992}...|
+------+--------------------+



In [42]:
# 추천 결과를 Python 객체로 받아오기
movies_lst = user_recs.collect()[0].recommendations

In [43]:
movies_lst

[Row(movieId=8327, rating=5.026991844177246),
 Row(movieId=5767, rating=4.95051908493042),
 Row(movieId=114554, rating=4.926124095916748),
 Row(movieId=7099, rating=4.860919952392578),
 Row(movieId=127098, rating=4.857874870300293)]

In [44]:
# movies_lst에 대한 데이터프레임 생성
recs_df = spark.createDataFrame(movies_lst)
recs_df.show()

+-------+-----------------+
|movieId|           rating|
+-------+-----------------+
|   8327|5.026991844177246|
|   5767| 4.95051908493042|
| 114554|4.926124095916748|
|   7099|4.860919952392578|
| 127098|4.857874870300293|
+-------+-----------------+



In [47]:
# [+] 영화 데이터에 대한 데이터프레임 생성

file = 'movies.csv'

movies_df = spark.read.csv(path + file, inferSchema=True, header = True)
movies_df.show()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/workspaces/bigdata-spark-lab/data/movies.csv.

In [ ]:
# [+] recs_df, movies_df 에 대한 Temporary View 생성
recs_df.createOrReplaceTempView('recommendations')
movies_df.createOrReplaceTempView('movies')

In [ ]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
spark.sql("""
    SELECT *
    FROM movies JOIN recommendations
    ON movies.movieID = recommendations.movieID
    ORDER BY rating DESC
""").show()

### **4. 유저 별 영화 추천 서비스를 간단하게 구현하기**
1. SQL문 작성
2. 영화 추천 함수 작성
3. 영화 추천 테스트

In [ ]:
# SQL JOIN 연산을 통해 추천된 영화 제목 받아오기
query = """
SELECT * 
FROM movies JOIN recommendations 
ON movies.movieID = recommendations.movieID
ORDER BY rating DESC
"""

In [ ]:
# 입력된 유저에 대한 영화 추천 함수
def get_recommendations(user_id, num_recs):
    users_df = spark.createDataFrame([user_id], IntegerType()).toDF('userID')
    users_recs_df = model.recommendForUserSubset(users_df, num_recs)
    recs_lst = users_recs_df.collect()[0].recommendations
    recs_df = spark.createDataFrame(recs_lst)
    recs_df.createOrReplaceTempView('recommendations')
    recommended_movies = spark.sql(query)
    
    return recommended_movies

In [ ]:
# 1번 유저에 대한 영화 5개 추천
recs = get_recommendations(1, 5)

In [ ]:
# 추천 결과 출력
recs.show()

In [ ]:
# toPandas(): Pandas 데이터프레임으로 출력
recs.toPandas()